<a href="https://colab.research.google.com/github/basit-02/ML-01_Starter/blob/main/work/notebooks/w03_feature_leakage_check.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/basit-02/ML-01_Starter/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector


We establish a localized connection to the FlyRank warehouse, extract the March 2026 training slice, and construct a clean feature vector `X`. We align historical DuckDB columns, impute `avg_position` where data is missing (0 becomes 100.0), and isolate our target proxy (`is_declining_proxy`).

In [1]:
import pandas as pd
import duckdb
import numpy as np
from google.colab import userdata

# Authenticate and load DuckDB data
hf_token = userdata.get('HF_TOKEN')
con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute(f"CREATE OR REPLACE SECRET hf_secret (TYPE HUGGINGFACE, TOKEN '{hf_token}');")

WAREHOUSE_PATH = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet"
query = f"SELECT * FROM '{WAREHOUSE_PATH}' WHERE month = '2026-03' LIMIT 30000"
df = con.execute(query).df()

# Align columns and handle Safe Data Imputation
rename_map = {"gsc_impressions": "impressions", "gsc_position": "avg_position", "adjusted_avg_position": "avg_position"}
df = df.rename(columns=rename_map)

df['avg_position'] = df['avg_position'].replace(0, 100.0).fillna(100.0) if 'avg_position' in df.columns else 100.0
df['days_since_last_update'] = df['days_since_last_update'].fillna(0) if 'days_since_last_update' in df.columns else 0
df['gsc_clicks'] = df['gsc_clicks'].fillna(0) if 'gsc_clicks' in df.columns else 1

# Define Proxy Label
df['is_declining_proxy'] = (df['gsc_clicks'] == 0).astype(int)

# Build feature matrix
raw_features = ['impressions', 'avg_position', 'ga4_sessions', 'ga4_total_engagement_sec', 'days_since_last_update']
for col in raw_features:
    if col not in df.columns:
        df[col] = 0

X = df[raw_features].fillna(0)
print(f"Feature vector X built with shape: {X.shape}")
X.head(3)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature vector X built with shape: (30000, 5)


,impressions,avg_position,ga4_sessions,ga4_total_engagement_sec,days_since_last_update
0,20,100.0,0,0,0
1,1,100.0,0,0,0
2,125,100.0,0,0,0


## 2. Feature notes (meaning, missing, categorical, available-when?)


*   **impressions:** The count of times the page appeared in search results. Missing values imputed to 0. Available prior to the proxy prediction moment (T-1 metric).
*   **avg_position:** The observed ranking position. Missing or 0 values are strictly imputed to 100.0 to represent "no visibility". Available prior to prediction.
*   **ga4_sessions & ga4_total_engagement_sec:** Traffic and engagement volume. Missing values imputed to 0. Available as trailing indicators.
*   **days_since_last_update:** Numeric count of days since editorial change. Missing values imputed to 0. Fully deterministic and available prior to prediction.

In [2]:
# Displaying data types and null counts to confirm clean feature extraction
feature_audit = pd.DataFrame({
    'Data Type': X.dtypes,
    'Missing Values': X.isna().sum(),
    'Exists T-1 (Before Prediction)': ['Yes'] * len(X.columns)
})
feature_audit

,Data Type,Missing Values,Exists T-1 (Before Prediction)
impressions,int64,0,Yes
avg_position,float64,0,Yes
ga4_sessions,Int64,0,Yes
ga4_total_engagement_sec,Int64,0,Yes
days_since_last_update,int64,0,Yes


## 3. The leakage hunt


Data leakage occurs if the model trains on features derived from the outcome it is trying to predict, or if it learns from existing heuristic rules rather than raw data. We systematically hunt for and isolate product flags (`health_score`, `quick_win`), label-derived trends, and the target proxy itself.

In [3]:
# Identify leaky columns based on the Honest Models framework
leaky_candidates = ['health_score', 'quick_win', 'is_declining_label', 'gsc_clicks', 'is_declining_proxy']
trend_columns = [col for col in df.columns if 'trend' in col.lower()]
full_drop_list = leaky_candidates + trend_columns

leaks_found = [col for col in full_drop_list if col in df.columns]

print("--- LEAKAGE AUDIT TRIGGERED ---")
print(f"Isolated and dropped the following {len(leaks_found)} leaky columns from training availability:")
for leak in leaks_found:
    print(f"- {leak}")

--- LEAKAGE AUDIT TRIGGERED ---
Isolated and dropped the following 2 leaky columns from training availability:
- gsc_clicks
- is_declining_proxy


## 4. What I excluded and why

*   **`health_score` & `quick_win`:** Excluded because these are pre-existing handwritten heuristic rules. Using them as features encodes the current product's logic into the model, forcing it to memorize the baseline rather than beat it.
*   **`is_declining_label` & `trend_*` columns:** Excluded because these are derived directly from the temporal trend of the proxy label. They contain future knowledge (data leakage) that would artificially inflate model performance during training but fail in production.
*   **`url` / `id`:** Excluded from the feature vector (used only for `GroupShuffleSplit`) to prevent the model from memorizing page-specific historical performance.

In [4]:
# Final confirmation of exclusion
assert len(set(X.columns).intersection(set(full_drop_list))) == 0, "CRITICAL ERROR: Leakage columns detected in feature vector X!"
print("System Check: Feature vector X is certified leak-free and public-safe.")

System Check: Feature vector X is certified leak-free and public-safe.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.